In [113]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

from pathlib import Path

# Opening cleaned PAROS dataset

In [114]:
CURRENT_DIRECTORY = Path.cwd().resolve()

# Find project root that contains datasets
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "PAROS_Dataset_Cleaned.csv"

if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_DATASET_PATH}")

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f"Loaded cleaned PAROS dataset: {df.shape}")
display(df.head(3))


Loaded cleaned PAROS dataset: (2076, 71)


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,Year,Call_Time,Shock_Time,Time_to_Defib
0,Ems,2014-01-01,238889.0,NaN,Transport Center,Dhoby Ghaut Mrt Level B1,59,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 22:28:12,2026-04-05 22:39:17,11.083333
1,Ems,2014-01-05,272018.0,NaN,Public/Commercial Building,Level 2,66,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 15:00:42,2026-04-05 15:16:49,16.116667
2,Ems,2014-01-07,760105.0,NaN,Street/Highway,Level 1,80,Years,Male,Indian,...,Admitted,Remains In Hospital At 30Th Day Post Arrest,NaN,4.0,4.0,NaN,2014,2026-04-05 12:05:46,2026-04-05 12:14:08,8.366667


# Standardize Reporting

In [115]:
def report_results(model_output, model_name="Model"):
    """
    Standardizes the reporting style for all versions.
    Uses .copy() and .loc to avoid ChainedAssignmentErrors.
    """
    # 1. Use .copy() to ensure 'summary' is an independent DataFrame
    summary = model_output.summary2().tables[1].copy()
    
    # 2. Use .loc[row_indexer, column_indexer] for explicit assignment
    summary.loc[:, 'OR'] = np.exp(summary['Coef.'])
    summary.loc[:, 'Lower_95_CI'] = np.exp(summary['Coef.'] - 1.96 * summary['Std.Err.'])
    summary.loc[:, 'Upper_95_CI'] = np.exp(summary['Coef.'] + 1.96 * summary['Std.Err.'])
    
    # 3. Clean and Rename Columns
    final_table = summary[['OR', 'Lower_95_CI', 'Upper_95_CI', 'P>|z|']]
    final_table.columns = ['Odds Ratio', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
    
    print(f"\n{'='*20} {model_name} Results {'='*20}")
    return final_table

# Define Binary Outcome (Survival to 30 Days)


In [116]:
outcome_cols = ['Outcome of patient', 'Patient status', 'Final status at scene']
df.loc[:, 'Outcome_String'] = df[outcome_cols].astype(str).agg(' '.join, axis=1)
survival_regex = r'Discharged Alive|Remains in hospital at 30th day'
df.loc[:, 'Survival_Binary'] = df['Outcome_String'].str.contains(survival_regex, case=False, regex=True).astype(int)

# Define Primary Predictor (Bystander AED applied)
- We map any variation of 'Yes' or '1' to 1, and everything else to 0

In [117]:
aed_col = 'Bystander AED applied'
df.loc[:, 'AED_Applied_Binary'] = df[aed_col].astype(str).str.contains('Yes|Applied|1', case=False, na=False).astype(int)

# Cleaned DataFrame

In [118]:
model_vars = ['Survival_Binary', 'AED_Applied_Binary', 'Time_to_Defib']
df_clean = df.dropna(subset=model_vars).copy()

# Grid Search

In [119]:
def run_piecewise_iteration(data, tau):
    # Segmented Time variables
    t_before = data['Time_to_Defib'].clip(upper=tau)
    t_after = (data['Time_to_Defib'] - tau).clip(lower=0)
    
    # Build Matrix X (Low-Level Style)
    X = pd.DataFrame({
        'const': 1.0,
        'AED': data['AED_Applied_Binary'],
        'Time_Before': t_before,
        'Time_After': t_after,
        'AED_x_Before': data['AED_Applied_Binary'] * t_before,
        'AED_x_After': data['AED_Applied_Binary'] * t_after
    })
    
    try:
        # Use 'bfgs' solver: more stable for clinical data with 'complete separation' issues
        # Increase maxiter to 500 to give the model room to converge
        model = sm.Logit(data['Survival_Binary'], X).fit(
            method='bfgs', 
            maxiter=500, 
            disp=0
        )
        
        # Check if the model actually converged
        if not model.mle_retvals['warnflag'] == 0:
            return np.inf, None
            
        return model.aic, model
    except:
        # If the math fails (Singular Matrix), return infinity
        return np.inf, None

# Check if we have survivors at late time points


In [120]:
late_survivors = df[(df['Time_to_Defib'] > 10) & (df['Survival_Binary'] == 1)]
print(f"Survivors after 10 mins: {len(late_survivors)}")
print(late_survivors['Bystander AED applied'].value_counts())

Survivors after 10 mins: 262
Bystander AED applied
No     243
Yes     19
Name: count, dtype: int64


# Grid Search

In [121]:
search_results = []
models_dict = {}

print("--- Stabilized Grid Search ---")
for t in range(3, 13):
    aic, model = run_piecewise_iteration(df_clean, t)
    
    # Only record results if AIC is not infinity (indicating convergence)
    if aic != np.inf:
        search_results.append({'tau': t, 'AIC': aic})
        models_dict[t] = model
        print(f"Minute {t}: AIC = {aic:.2f}")
    else:
        print(f"Minute {t}: Optimization failed to converge. Skipping.")

# Find the winner
discovery_df = pd.DataFrame(search_results)
best_tau = discovery_df.loc[discovery_df['AIC'].idxmin(), 'tau']
print(f"\nValidated Optimal Breakpoint (tau): {best_tau} minutes")

--- Stabilized Grid Search ---
Minute 3: AIC = 1849.86
Minute 4: AIC = 1849.64
Minute 5: AIC = 1850.55
Minute 6: AIC = 1850.89
Minute 7: AIC = 1851.48
Minute 8: AIC = 1852.09
Minute 9: AIC = 1852.93
Minute 10: AIC = 1853.37
Minute 11: AIC = 1853.65
Minute 12: AIC = 1853.82

Validated Optimal Breakpoint (tau): 4 minutes


# Extract Results and Convert to Odds Ratios (OR)

In [123]:
winner = models_dict[best_tau]
v3_final_report = report_results(winner, "Version 3: Piecewise (tau=5)")

display(v3_final_report)


==================== Version 3: Piecewise (tau=5) Results ====================


,Odds Ratio,Lower 95% CI,Upper 95% CI,p-value
const,3.549771e+02,2.458538e-03,5.125353e+07,3.326599e-01
AED,5.179091e-14,5.968115e-319,4.494372e+291,9.319478e-01
Time_Before,2.062556e-01,1.048730e-02,4.056468e+00,2.989606e-01
Time_After,8.721812e-01,8.359064e-01,9.100301e-01,2.793003e-10
AED_x_Before,2.027467e+03,1.180180e-73,3.483048e+79,9.322441e-01
AED_x_After,1.059042e+00,9.412779e-01,1.191539e+00,3.401881e-01
